In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import io
from google.colab import files

In [ ]:
uploaded = files.upload()

Saving train_preprocessed.csv to train_preprocessed.csv


In [ ]:
train_df = pd.read_csv(io.BytesIO(uploaded['train_preprocessed.csv']))

In [ ]:
uploadd = files.upload()

Saving test_preprocessed.csv to test_preprocessed.csv


In [ ]:
test_df = pd.read_csv(io.BytesIO(uploadd['test_preprocessed.csv']))

In [ ]:
# Drop rows with missing target
train_df = train_df.dropna(subset=['spend_category'])

X = train_df.drop(columns=['trip_id', 'spend_category'])
y = train_df['spend_category']

# Prepare Test Data
test_ids = test_df['trip_id']
X_test = test_df.drop(columns=['trip_id', 'spend_category'])
X_test = X_test[X.columns]

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train GMM Classifiers (one GMM model for each unique class)
classes = sorted(y_train.unique())
models = {}
priors = {}

print("Training GMMs...")
for c in classes:
    # Select only data belonging to class 'c'
    X_c = X_train[y_train == c]

    # Calculate prior probability (Class Frequency / Total Count)
    priors[c] = len(X_c) / len(X_train)

    gmm = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
    gmm.fit(X_c)
    models[c] = gmm

Training GMMs...


In [ ]:
# Prediction Logic
def predict_gmm(X, models, priors):
    scores = np.zeros((X.shape[0], len(models)))  # log-likelihood score for each class

    for idx, c in enumerate(sorted(models.keys())):
        scores[:, idx] = models[c].score_samples(X) + np.log(priors[c])  # add the log of the prior to calculate the posterior

    # Pick the class index with the highest score
    pred_indices = np.argmax(scores, axis=1)
    # Map index back to original class label
    return [sorted(models.keys())[i] for i in pred_indices]

In [ ]:
# Evaluate
y_pred_val = predict_gmm(X_val, models, priors)

print(f"Validation Accuracy: {accuracy_score(y_val, y_pred_val):.4f}")
print("Classification Report:")
print(classification_report(y_val, y_pred_val))

Validation Accuracy: 0.5852
Classification Report:
              precision    recall  f1-score   support

         0.0       0.63      0.94      0.75      1174
         1.0       0.68      0.20      0.31      1076
         2.0       0.36      0.61      0.45       274

    accuracy                           0.59      2524
   macro avg       0.55      0.58      0.50      2524
weighted avg       0.62      0.59      0.53      2524



In [ ]:
# Predict on test.csv
test_predictions = predict_gmm(X_test, models, priors)

submission = pd.DataFrame({
    'trip_id': test_ids,
    'spend_category': test_predictions
})

print("\nSubmission Preview:")
print(submission.head())
submission.to_csv('gmm_submission.csv', index=False)


Submission Preview:
           trip_id  spend_category
0  tour_id8gzpck76             2.0
1  tour_idow1zxkou             0.0
2  tour_idue7esfqz             0.0
3  tour_idnj3mjzpb             0.0
4  tour_ida3us5yk2             0.0
